**PROJECT NAME: GROUPDNA**

**NAME: G.V.S.NEHA SREE**

**ROLL NUMBER:**

**BATCH: JUNE 2026**

**DATE: 29-06-2026**

In [39]:
import numpy as np
from datetime import datetime

# --- Feature 1: Parser ---
parsed_messages = []
system_msg_count = 0
media_omitted_count = 0
deleted_msg_count = 0

media_per_person = {}
deleted_per_person = {}

file_path = 'hostel_bois.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

for line in lines:
    line = line.strip()
    if not line:
        continue

    is_new_message_start = len(line) >= 17 and line[2] == '/' and line[5] == '/' and line[8] == ',' and line[12] == ':'

    if not is_new_message_start:
        if parsed_messages:
            parsed_messages[-1]['text'] += " " + line
        continue

    try:
        timestamp_full = line[0:17]
        remaining_content = line[17:].strip()

        if ':' not in remaining_content:
            system_msg_count += 1
            continue

        if remaining_content.startswith('- '):
            sender_and_text_part = remaining_content[2:]
        else:
            sender_and_text_part = remaining_content

        sender, text = sender_and_text_part.split(':', 1)
        sender = sender.strip()
        text = text.strip()

        if sender not in media_per_person: media_per_person[sender] = 0
        if sender not in deleted_per_person: deleted_per_person[sender] = 0

        if text == "<Media omitted>":
            media_omitted_count += 1
            media_per_person[sender] += 1
            parsed_messages.append({'timestamp': timestamp_full, 'sender': sender, 'text': text, 'type': 'media'})
        elif text == "This message was deleted":
            deleted_msg_count += 1
            deleted_per_person[sender] += 1
            parsed_messages.append({'timestamp': timestamp_full, 'sender': sender, 'text': text, 'type': 'deleted'})
        else:
            parsed_messages.append({'timestamp': timestamp_full, 'sender': sender, 'text': text, 'type': 'real'})

    except Exception as e:
        system_msg_count += 1

# --- Feature 2 & 3: Group Overview Processing ---
total_messages = len(parsed_messages)
actual_unique_senders = sorted(list(set([msg['sender'] for msg in parsed_messages])))

unique_participants = actual_unique_senders
person_to_idx = {name: idx for idx, name in enumerate(unique_participants)}

msg_counts_per_person = {}
for msg in parsed_messages:
    sender = msg['sender']
    msg_counts_per_person[sender] = msg_counts_per_person.get(sender, 0) + 1

sorted_ranking = sorted(msg_counts_per_person.items(), key=lambda x: x[1], reverse=True)

# Heatmap Setup
heatmap_matrix = np.zeros((len(unique_participants), 24))
day_activity = {}
hour_activity = {}

chronological_msgs = []

for msg in parsed_messages:
    ts = msg['timestamp']
    if ',' in ts:
        date_part, time_part = ts.split(',', 1)
        hour_part = time_part.strip().split(':', 1)[0]
    else:
        date_part = ts[:8]
        hour_part = ts[9:11]

    day_activity[date_part] = day_activity.get(date_part, 0) + 1
    hour_activity[hour_part] = hour_activity.get(hour_part, 0) + 1

    try:
        dt_obj = datetime.strptime(ts.split(' -')[0].strip(), '%d/%m/%y, %H:%M')
        hour = dt_obj.hour
        if 0 <= hour <= 23:
            p_idx = person_to_idx[msg['sender']]
            heatmap_matrix[p_idx, hour] += 1
        chronological_msgs.append({'sender': msg['sender'], 'datetime': dt_obj, 'type': msg['type'], 'text': msg['text']})
    except ValueError:
        continue

busiest_date = max(day_activity, key=day_activity.get)
busiest_hour = max(hour_activity, key=hour_activity.get)

# --- Feature 6 Setup: Response Speed & Silent Streaks ---
unique_participants_in_chrono = unique_participants
replies_gaps = {name: [] for name in unique_participants_in_chrono}
for i in range(1, len(chronological_msgs)):
    prev_msg = chronological_msgs[i-1]
    curr_msg = chronological_msgs[i]
    if curr_msg['sender'] != prev_msg['sender']:
        gap_seconds = (curr_msg['datetime'] - prev_msg['datetime']).total_seconds()
        if gap_seconds > 0:
            replies_gaps[curr_msg['sender']].append(gap_seconds)

start_date = datetime(2024, 4, 1)
end_date = datetime(2024, 5, 30)
total_days_range = (end_date - start_date).days + 1

member_active_days = {name: set() for name in unique_participants_in_chrono}
for msg in chronological_msgs:
    day_str = msg['datetime'].strftime('%Y-%m-%d')
    member_active_days[msg['sender']].add(day_str)

# --- Feature 7 & 8 Setup: Personality Archetypes Assignment ---
spammer_bursts = {name: [] for name in unique_participants}
current_burst = 1
for i in range(1, len(parsed_messages)):
    if parsed_messages[i]['sender'] == parsed_messages[i-1]['sender']:
        current_burst += 1
    else:
        spammer_bursts[parsed_messages[i-1]['sender']].append(current_burst)
        current_burst = 1
spammer_bursts[parsed_messages[-1]['sender']].append(current_burst)

caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
lol_keywords = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
archetype_scores = {name: {} for name in unique_participants}

for name in unique_participants:
    p_idx = person_to_idx[name]
    user_msgs = [m for m in parsed_messages if m['sender'] == name]
    real_msgs = [m for m in user_msgs if m['type'] == 'real']

    # THE SPAMMER
    bursts = spammer_bursts[name]
    archetype_scores[name]['THE SPAMMER'] = sum(bursts)/len(bursts) if bursts else 0

    # THE GROUP MOM
    caring_count = 0
    for m in real_msgs:
        for word in caring_keywords:
            if word in m['text'].lower(): caring_count += 1
    archetype_scores[name]['THE GROUP MOM'] = caring_count

    # THE NIGHT OWL
    night_msgs = heatmap_matrix[p_idx, 23] + np.sum(heatmap_matrix[p_idx, 0:5])
    total_user_msgs = len(user_msgs)
    archetype_scores[name]['THE NIGHT OWL'] = (night_msgs / total_user_msgs * 100) if total_user_msgs > 0 else 0

    # THE STORYTELLER
    total_words = sum([len(m['text'].split()) for m in real_msgs])
    archetype_scores[name]['THE STORYTELLER'] = (total_words / len(real_msgs)) if real_msgs else 0

    # THE DRAMA QUEEN
    dq_count = 0
    for m in real_msgs:
        txt = m['text']
        if (txt.isupper() and len(txt) >= 3) or (txt.count('!') >= 2):
            dq_count += 1
    archetype_scores[name]['THE DRAMA QUEEN'] = (dq_count / len(user_msgs) * 100) if user_msgs else 0

    # THE GHOST
    silent_days = total_days_range - len(member_active_days[name])
    archetype_scores[name]['THE GHOST'] = (silent_days / total_days_range * 100)

    # THE COMEDIAN
    lol_count = 0
    for m in real_msgs:
        for term in lol_keywords:
            if term in m['text'].lower(): lol_count += 1
    archetype_scores[name]['THE COMEDIAN'] = (lol_count / len(user_msgs) * 100) if user_msgs else 0

    # THE QUESTION MASTER
    qm_count = sum([1 for m in real_msgs if m['text'].endswith('?')])
    archetype_scores[name]['THE QUESTION MASTER'] = (qm_count / len(user_msgs) * 100) if user_msgs else 0

assigned_archetypes = {}
archetype_hierarchy = ['THE SPAMMER', 'THE NIGHT OWL', 'THE DRAMA QUEEN', 'THE STORYTELLER', 'THE GHOST', 'THE GROUP MOM', 'THE COMEDIAN', 'THE QUESTION MASTER']

for target_archetype in archetype_hierarchy:
    best_candidate = None
    best_score = -1
    for name in unique_participants:
        if name in assigned_archetypes:
            continue
        score = archetype_scores[name][target_archetype]
        if score > best_score:
            best_score = score
            best_candidate = name
    if best_candidate:
        assigned_archetypes[best_candidate] = target_archetype

print("============================================================")
print("GROUPDNA REPORT -- Hostel Bois 4ever")
print(f"60 days   . {total_messages} messages  .{len(unique_participants)} members")
print("============================================================")
print(f"\nPeriod         : 01 April 2024 to 30 May 2024 (60 days)")
print(f"Busiest day  : {busiest_date} ({day_activity[busiest_date]} messages)")
print(f"Busiest hour : {busiest_hour}:00 (Total count: {hour_activity[busiest_hour]} messages)")

print("\nMESSAGES PER PERSON")
for person, count in sorted_ranking:
    percentage = (count / total_messages) * 100
    bar_len = int(percentage / 5)
    bar = '█' * bar_len
    print(f"{person:<8} : {bar:<20} {count:>3} ({percentage:.1f}%)")

print("\nACTIVITY HEATMAP (messages by hour)")
print("         00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")
for name in unique_participants:
    p_idx = person_to_idx[name]
    row_str = f"{name:<8} "
    max_val = np.max(heatmap_matrix[p_idx, :])
    for h in range(24):
        val = heatmap_matrix[p_idx, h]
        if max_val == 0: shade = '  '
        elif val <= max_val * 0.25: shade = '. '
        elif val <= max_val * 0.50: shade = '░ '
        elif val <= max_val * 0.75: shade = '▒ '
        else: shade = '█ '
        row_str += shade
    print(row_str)

# --- Top Words ---
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'my', 'you', 'this', 'that'}
word_counts = {}
punctuation_chars = '.,!?;:"()[]{}~*_-/\\'
for msg in parsed_messages:
    if msg['type'] != 'real': continue
    words = msg['text'].lower().split()
    for word in words:
        word = word.strip(punctuation_chars)
        if word and word not in stop_words:
            word_counts[word] = word_counts.get(word, 0) + 1
top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nTHIS GROUP'S FAVOURITE WORDS")
max_word_count = top_words[0][1] if top_words else 1
for word, count in top_words:
    bar_len = int((count / max_word_count) * 20)
    bar = '█' * bar_len
    print(f"{word:<8} {bar:<20} {count}")


print("\nRESPONSE PATTERNS")
avg_gaps = {}
for name in unique_participants_in_chrono:
    gaps = replies_gaps[name]
    avg_gaps[name] = (sum(gaps) / len(gaps)) if gaps else float('inf')
valid_gaps = {k: v for k, v in avg_gaps.items() if v != float('inf')}

if valid_gaps:
    fastest_user = min(valid_gaps, key=valid_gaps.get)
    slowest_user = max(valid_gaps, key=valid_gaps.get)
    def format_duration(seconds):
        minutes = seconds / 60
        if minutes < 60: return f"{minutes:.1f} minutes"
        else: return f"{(minutes / 60):.1f} hours"
    print(f"  Fastest replier : {fastest_user:<7} (avg {format_duration(valid_gaps[fastest_user])})")
    print(f"  Slowest replier : {slowest_user:<7} (avg {format_duration(valid_gaps[slowest_user])})")

print("\nLONGEST SILENT STREAKS")
longest_silent_streaks = {}
for name in unique_participants_in_chrono:
    max_streak, current_streak = 0, 0
    current_streak_start, best_start_date, best_end_date = None, None, None
    for d_offset in range(total_days_range):
        curr_day_dt = start_date + np.timedelta64(d_offset, 'D').astype(object)
        curr_day_str = curr_day_dt.strftime('%Y-%m-%d')
        if curr_day_str not in member_active_days[name]:
            if current_streak == 0: current_streak_start = curr_day_dt
            current_streak += 1
        else:
            if current_streak > max_streak:
                max_streak = current_streak
                best_start_date = current_streak_start
                best_end_date = curr_day_dt - np.timedelta64(1, 'D').astype(object)
            current_streak = 0
    if current_streak > max_streak:
        max_streak = current_streak
        best_start_date = current_streak_start
        best_end_date = end_date
    longest_silent_streaks[name] = {'days': max_streak, 'start': best_start_date, 'end': best_end_date}

for name in unique_participants_in_chrono:
    st_data = longest_silent_streaks[name]
    days = st_data['days']
    if days > 0 and st_data['start'] and st_data['end']:
        start_str = st_data['start'].strftime('%d %b').lstrip('0')
        end_str = st_data['end'].strftime('%d %b').lstrip('0')
        date_range_str = f"({start_str} - {end_str})"
    else: date_range_str = ""
    range_str = f" {date_range_str}" if days > 3 else ""
    print(f"  {name:<7} : {days:>2} days{range_str}")
print("\nPERSONALITY ARCHETYPES")
def format_archetype_details(name, archetype):
    score = archetype_scores[name][archetype]
    if archetype == 'THE SPAMMER': return f"(avg {score:.1f} msgs in a row)"
    elif archetype == 'THE GROUP MOM': return f"(caring keyword score: {int(score)})"
    elif archetype == 'THE NIGHT OWL': return f"({score:.1f}% msgs between 23h-04h)"
    elif archetype == 'THE STORYTELLER': return f"(avg {score:.1f} words per msg)"
    elif archetype == 'THE DRAMA QUEEN': return f"({score:.1f}% ALL-CAPS messages)"
    elif archetype == 'THE GHOST':
        s_days = total_days_range - len(member_active_days[name])
        return f"(silent on {s_days} of {total_days_range} days)"
    elif archetype == 'THE COMEDIAN': return f"({score:.1f}% comedic/reaction msgs)"
    elif archetype == 'THE QUESTION MASTER': return f"({score:.1f}% inquisitive messages)"
    return ""

for name in unique_participants:
    arch = assigned_archetypes.get(name, "THE CHAMELEON")
    details = format_archetype_details(name, arch)
    print(f"  {name:<7}  →  {arch:<19} {details}")

print("\n" + "=" * 70)
print(f"  Generated by GroupDNA   •   Built with Python + NumPy")
print("=" * 70)

GROUPDNA REPORT -- Hostel Bois 4ever
60 days   . 3174 messages  .6 members

Period         : 01 April 2024 to 30 May 2024 (60 days)
Busiest day  : 04/05/24 (76 messages)
Busiest hour : 18:00 (Total count: 248 messages)

MESSAGES PER PERSON
Rahul    : ██████               953 (30.0%)
Priya    : ████                 718 (22.6%)
Neha     : ████                 635 (20.0%)
Aman     : ███                  490 (15.4%)
Karan    : ██                   354 (11.2%)
Vikas    :                       24 (0.8%)

ACTIVITY HEATMAP (messages by hour)
         00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman     ▒ █ ▒ ▒ █ . . . . . . . . . . . . . . . . . . ▒ 
Karan    . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
Neha     . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
Priya    . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
Rahul    . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
Vikas    . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 

THIS GROUP'S FAVOURITE WORDS
w

**Balancing these multi-conditional tracking layers to capture accurate date ranges is highly complex.**

**I would use text patterns (Regex) for cleaner parsing, word percentages for fairer archetypes, and date gaps instead of checking every single calendar day to make silent streak calculations instant.**